# 論文再現バックテスト + 検証レイヤー

**対象論文**: 中川他 (2025)「部分空間正則化付き主成分分析を用いた日米業種リードラグ投資戦略」
**ユニバース**: 26 銘柄固定（米国 9・日本 17、XLC / XLRE 除外） — 要件定義書 v3 §2.1
**判定方針**: 合否は論文の R/R 一致ではなく、**IC・分位スプレッド・ルックアヘッド監査**（要件定義書 v3 §4）で判定する。R/R 値はあくまで参考値。

## 構成

| サブセクション | 内容 |
|---|---|
| 2A（本ファイル先頭） | データ取得・共通営業日インターセクション |
| 2B | リターン定義（CC / OC） |
| 2C | 事前相関 C_full・バックテストループ |
| 2D | 検証レイヤー（IC・分位スプレッド・ルックアヘッド監査） — 合否判定の核心 |
| 2E | パフォーマンス評価と論文比較 |


In [1]:
# SECTION 2A — Imports & settings
import os
import sys
import time
import warnings
from datetime import datetime, date

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

# Make src/ importable (this notebook lives in notebooks/)
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from src.universe import (
    ALL_TICKERS, US_TICKERS, JP_TICKERS,
    N_US, N_JP, N_TOTAL,
    get_universe_masks,
    US_SECTOR_NAMES, JP_SECTOR_NAMES,
)
from src.pca_engine import (
    build_prior_subspace,
    build_target_correlation,
    compute_regularized_correlation,
    extract_top_eigenvectors,
    compute_lead_lag_signal,
)

# Reproducibility
RNG_SEED = 20260515
rng = np.random.default_rng(RNG_SEED)

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning, module="yfinance")

print("Repo root:", REPO_ROOT)
print("Python   :", sys.version.split()[0])
print("numpy    :", np.__version__)
print("pandas   :", pd.__version__)
print("yfinance :", yf.__version__)


Repo root: /Users/tamairyoga/Desktop/MacBook Pro/システム開発/sector-leadlag
Python   : 3.11.15
numpy    : 2.4.4
pandas   : 3.0.3
yfinance : 1.3.0


In [2]:
# SECTION 2A — Universe check
assert N_US == 9 and N_JP == 17 and N_TOTAL == 26, "Universe must be 9 + 17 = 26 (v3)"
print(f"Universe: N_US={N_US}, N_JP={N_JP}, total={N_TOTAL}")
print(f"US_TICKERS ({N_US}):", US_TICKERS)
print(f"JP_TICKERS ({N_JP}):", JP_TICKERS)

masks = get_universe_masks()
for k, m in masks.items():
    on = [t for t, v in zip(US_TICKERS if k.startswith("us") else JP_TICKERS, m) if v]
    print(f"  {k:13s} shape={m.shape}, true_count={int(m.sum())} -> {on}")


Universe: N_US=9, N_JP=17, total=26
US_TICKERS (9): ['XLB', 'XLE', 'XLF', 'XLI', 'XLK', 'XLP', 'XLU', 'XLV', 'XLY']
JP_TICKERS (17): ['1617.T', '1618.T', '1619.T', '1620.T', '1621.T', '1622.T', '1623.T', '1624.T', '1625.T', '1626.T', '1627.T', '1628.T', '1629.T', '1630.T', '1631.T', '1632.T', '1633.T']
  us_cyclical   shape=(9,), true_count=3 -> ['XLB', 'XLE', 'XLF']
  us_defensive  shape=(9,), true_count=4 -> ['XLK', 'XLP', 'XLU', 'XLV']
  jp_cyclical   shape=(17,), true_count=4 -> ['1618.T', '1625.T', '1629.T', '1631.T']
  jp_defensive  shape=(17,), true_count=4 -> ['1617.T', '1621.T', '1627.T', '1630.T']


In [3]:
# SECTION 2A — Data fetch (auto_adjust=True, Open & Close, with cache & retry)
#
# Cache format: a single MultiIndex CSV with columns (field, ticker) where
# field ∈ {"Open", "Close"} and ticker iterates ALL_TICKERS. The CSV header
# spans two rows so pandas can round-trip the MultiIndex losslessly.

START_DATE = pd.Timestamp("2010-01-01")
END_DATE   = pd.Timestamp("2025-12-31")
TODAY      = pd.Timestamp(date.today())
# We can't fetch data beyond today; the effective upper bound is min(END_DATE, today).
EFFECTIVE_END = min(END_DATE, TODAY)
# Tolerate up to 7 calendar-day staleness in the cache (weekends / holidays).
STALENESS_TOL = pd.Timedelta(days=7)

CACHE_PATH = os.path.join(REPO_ROOT, "data", "cache_prices.csv")
os.makedirs(os.path.dirname(CACHE_PATH), exist_ok=True)


def _download_with_retry(tickers, start, end, max_retries=3):
    last_err = None
    for attempt in range(max_retries):
        try:
            df = yf.download(
                tickers,
                start=start,
                end=end,
                auto_adjust=True,
                progress=False,
                threads=True,
                group_by="column",
            )
            if df is None or df.empty:
                raise RuntimeError("yfinance returned an empty DataFrame")
            return df
        except Exception as exc:
            last_err = exc
            wait = 2 ** attempt  # 1s -> 2s -> 4s
            print(f"  [retry {attempt + 1}/{max_retries}] {exc!r}; sleeping {wait}s")
            time.sleep(wait)
    raise RuntimeError(f"yfinance download failed after {max_retries} retries: {last_err}")


def _load_cache(path):
    # First two rows of the file are the MultiIndex column header
    df = pd.read_csv(path, header=[0, 1], index_col=0, parse_dates=True)
    df.columns = pd.MultiIndex.from_tuples(df.columns, names=["field", "ticker"])
    return df


def _save_cache(df, path):
    df.to_csv(path)


raw = None
use_cache = os.path.exists(CACHE_PATH)
if use_cache:
    try:
        cached = _load_cache(CACHE_PATH)
        cached_start = cached.index.min()
        cached_end   = cached.index.max()
        covers_start   = cached_start <= START_DATE
        covers_end     = cached_end   >= (EFFECTIVE_END - STALENESS_TOL)
        covers_tickers = set(ALL_TICKERS).issubset(set(cached.columns.get_level_values("ticker")))
        if covers_start and covers_end and covers_tickers:
            print(f"Using cache: {CACHE_PATH}")
            print(f"  range  : {cached_start.date()} … {cached_end.date()}, rows={len(cached)}")
            raw = cached
        else:
            reason = []
            if not covers_start:   reason.append(f"start>{START_DATE.date()}")
            if not covers_end:     reason.append(f"end<{(EFFECTIVE_END - STALENESS_TOL).date()}")
            if not covers_tickers: reason.append("tickers missing")
            print(f"Cache exists but is stale ({', '.join(reason)}); refetching.")
            use_cache = False
    except Exception as exc:
        print(f"Cache read failed ({exc!r}); refetching.")
        use_cache = False

if not use_cache or raw is None:
    # yfinance's `end` is exclusive; pad by 1 day so we get the latest available bar.
    fetch_end = (EFFECTIVE_END + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    fetch_start = START_DATE.strftime("%Y-%m-%d")
    print(f"Downloading {len(ALL_TICKERS)} tickers from yfinance: {fetch_start} … {fetch_end}")
    downloaded = _download_with_retry(ALL_TICKERS, fetch_start, fetch_end)

    # yfinance returns a MultiIndex (field, ticker). Keep only Open & Close.
    if isinstance(downloaded.columns, pd.MultiIndex):
        fields = downloaded.columns.get_level_values(0).unique().tolist()
        if "Open" not in fields or "Close" not in fields:
            raise RuntimeError(f"Expected Open & Close columns; got fields={fields}")
        open_df  = downloaded["Open"][ALL_TICKERS].copy()
        close_df = downloaded["Close"][ALL_TICKERS].copy()
    else:
        # Single-ticker shortcut (shouldn't happen for 26 tickers)
        raise RuntimeError("Unexpected single-level columns from yfinance")

    raw = pd.concat(
        {"Open": open_df, "Close": close_df},
        axis=1,
        names=["field", "ticker"],
    )
    raw.index.name = "Date"
    _save_cache(raw, CACHE_PATH)
    print(f"Cache written: {CACHE_PATH}  ({len(raw)} rows)")

# Split into per-field DataFrames for downstream use
open_raw  = raw["Open"][ALL_TICKERS].copy()
close_raw = raw["Close"][ALL_TICKERS].copy()
open_raw.index  = pd.to_datetime(open_raw.index)
close_raw.index = pd.to_datetime(close_raw.index)

print(f"open_raw  shape: {open_raw.shape}")
print(f"close_raw shape: {close_raw.shape}")
print(f"date range     : {close_raw.index.min().date()} … {close_raw.index.max().date()}")


Cache exists but is stale (start>2010-01-01); refetching.
Cache written: /Users/tamairyoga/Desktop/MacBook Pro/システム開発/sector-leadlag/data/cache_prices.csv  (4160 rows)
open_raw  shape: (4160, 26)
close_raw shape: (4160, 26)
date range     : 2010-01-04 … 2025-12-31


In [4]:
# SECTION 2A — Common business-day intersection (requirements_v3 §3.2)
# Rule: keep only dates where every one of the 26 tickers has a non-null Close.
# Align Open to that same date set.

non_null_close = close_raw.notna().all(axis=1)
common_dates = close_raw.index[non_null_close]

close_df = close_raw.loc[common_dates].copy()
open_df  = open_raw.loc[common_dates].copy()

# Open may still have NaN even when Close is non-null (rare). Audit it.
open_missing = open_df.isna().sum().sum()
if open_missing > 0:
    # Drop those rows too (consistent intersection between Open & Close)
    both_non_null = open_df.notna().all(axis=1) & close_df.notna().all(axis=1)
    common_dates = close_df.index[both_non_null]
    close_df = close_df.loc[common_dates].copy()
    open_df  = open_df.loc[common_dates].copy()
    print(f"NOTE: {open_missing} Open cells were NaN; intersected with Open availability too.")

N_BIZ = len(common_dates)
print(f"採用期間：{common_dates.min().date()} 〜 {common_dates.max().date()}, 共通営業日数：{N_BIZ}")

# Audit: consecutive NaN runs in the *raw* close per ticker (before intersection)
def _max_consecutive_nan_run(s: pd.Series) -> int:
    nan = s.isna().astype(int).values
    best = run = 0
    for x in nan:
        run = run + 1 if x else 0
        if run > best:
            best = run
    return best

gap_report = {t: _max_consecutive_nan_run(close_raw[t]) for t in ALL_TICKERS}
worst = sorted(gap_report.items(), key=lambda kv: kv[1], reverse=True)[:10]
print("\nMax consecutive NaN runs in raw Close (top 10):")
for t, n in worst:
    flag = "  ← ⚠️ ≥3" if n >= 3 else ""
    print(f"  {t:8s}: {n:3d} days{flag}")

flagged = [t for t, n in gap_report.items() if n >= 3]
if flagged:
    print(f"\n⚠️  WARNING: tickers with ≥3 consecutive NaN in raw Close: {flagged}")
    print("    These are tolerated (intersection removes them) but worth knowing.")
else:
    print("\nOK: no ticker has a ≥3-day consecutive NaN run in raw Close.")


採用期間：2010-01-04 〜 2025-12-30, 共通営業日数：3796

Max consecutive NaN runs in raw Close (top 10):
  1617.T  :   6 days  ← ⚠️ ≥3
  1618.T  :   6 days  ← ⚠️ ≥3
  1619.T  :   6 days  ← ⚠️ ≥3
  1620.T  :   6 days  ← ⚠️ ≥3
  1621.T  :   6 days  ← ⚠️ ≥3
  1622.T  :   6 days  ← ⚠️ ≥3
  1623.T  :   6 days  ← ⚠️ ≥3
  1624.T  :   6 days  ← ⚠️ ≥3
  1625.T  :   6 days  ← ⚠️ ≥3
  1626.T  :   6 days  ← ⚠️ ≥3

⚠️  WARNING: tickers with ≥3 consecutive NaN in raw Close: ['1617.T', '1618.T', '1619.T', '1620.T', '1621.T', '1622.T', '1623.T', '1624.T', '1625.T', '1626.T', '1627.T', '1628.T', '1629.T', '1630.T', '1631.T', '1632.T', '1633.T']
    These are tolerated (intersection removes them) but worth knowing.


## セル 5 の結果を読む

- **採用期間**：おおむね 2010-01-04 〜 2025 年後半（取得時点で利用可能な営業日まで）をカバーしているはず。
  XLC（2018 年上場）と XLRE（2015 年上場）を除外しているため、v2 のように 2018 年以降まで短縮されることは無い。
- **共通営業日数**：日米同時取引日のみのため、年あたり概ね 240 日前後 × 15〜16 年 ≒ 3,600〜3,800 日が想定範囲。
  もしこれを大きく下回る（例: 1,000 日台に短縮）場合は、26 銘柄以外が混入していないか、もしくは特定銘柄の長期欠損を疑う。
- **連続 3 営業日以上の欠損**：警告として可視化する。インターセクション後は除外されるが、原因（祝日連続・取引停止・配信遅延）を把握しておく。

問題なければ SECTION 2B（リターン定義）に進む。


In [5]:
# SECTION 2B — Close-to-Close returns (PCA estimation input)
# r_cc[i, t] = Close[i, t] / Close[i, t-1] - 1
# auto_adjust=True なので Close は配当・分割調整済み（要件定義書 v3 §2.4）

cc_returns = close_df.pct_change()  # shape (N_BIZ, 26), 最初の行は NaN

assert cc_returns.shape == (N_BIZ, N_TOTAL), (
    f"cc_returns shape {cc_returns.shape} != expected ({N_BIZ}, {N_TOTAL})"
)
assert cc_returns.iloc[0].isna().all(), "First row of cc_returns must be all NaN"
assert cc_returns.iloc[1:].notna().all().all(), (
    "cc_returns has NaN beyond the first row — inspect close_df"
)

# Outlier audit (≥50% daily move). Requirements v3 §3.3: warn only, do not stop.
extreme_cc = (cc_returns.abs() > 0.50).sum().sort_values(ascending=False)
n_extreme_cc = int(extreme_cc.sum())
print(f"cc_returns shape: {cc_returns.shape}")
print(f"  rows with ≥1 ticker |CC| > 50%: {int((cc_returns.abs() > 0.50).any(axis=1).sum())}")
print(f"  total cell-count of |CC| > 50%: {n_extreme_cc}")
if n_extreme_cc > 0:
    nz = extreme_cc[extreme_cc > 0]
    print(f"  tickers with ≥1 extreme CC: {nz.to_dict()}")
else:
    print("  no extreme CC return detected (>50%).")


cc_returns shape: (3796, 26)
  rows with ≥1 ticker |CC| > 50%: 2
  total cell-count of |CC| > 50%: 2
  tickers with ≥1 extreme CC: {'1629.T': 2}


In [6]:
# SECTION 2B — Open-to-Close returns (strategy P/L, Japan side only)
# r_oc[j, t] = Close[j, t] / Open[j, t] - 1   ※ 同一営業日 t の調整後始値→終値
# 戦略は時点 t に米国 CC で生成したシグナルを、日本側 t+1 の OC で執行・評価する。
# v2 で使っていた「日本側 CC リターンで近似」は廃止（要件定義書 v3 §2.4）。

jp_open  = open_df[JP_TICKERS]
jp_close = close_df[JP_TICKERS]
oc_returns_jp = jp_close / jp_open - 1.0

assert oc_returns_jp.shape == (N_BIZ, N_JP), (
    f"oc_returns_jp shape {oc_returns_jp.shape} != expected ({N_BIZ}, {N_JP})"
)
n_nan_oc = int(oc_returns_jp.isna().sum().sum())
assert n_nan_oc == 0, (
    f"oc_returns_jp must not contain NaN (same-day Open/Close); "
    f"found {n_nan_oc} NaN cells — inspect open_df / close_df alignment."
)

# Outlier audit (Requirements v3 §3.3)
extreme_oc = (oc_returns_jp.abs() > 0.50).sum().sort_values(ascending=False)
n_extreme_oc = int(extreme_oc.sum())
print(f"oc_returns_jp shape: {oc_returns_jp.shape}")
print(f"  rows with ≥1 ticker |OC| > 50%: {int((oc_returns_jp.abs() > 0.50).any(axis=1).sum())}")
print(f"  total cell-count of |OC| > 50%: {n_extreme_oc}")
if n_extreme_oc > 0:
    nz = extreme_oc[extreme_oc > 0]
    print(f"  tickers with ≥1 extreme OC: {nz.to_dict()}")
else:
    print("  no extreme OC return detected (>50%).")

# Quick descriptive stats (mean & std, annualised with 252)
daily_mean = oc_returns_jp.mean()
daily_std  = oc_returns_jp.std()
print("\nDaily OC return summary (Japan, 17 tickers):")
print(f"  mean (daily) min / max     : {daily_mean.min():.6f} / {daily_mean.max():.6f}")
print(f"  std  (daily) min / max     : {daily_std.min():.6f} / {daily_std.max():.6f}")
print(f"  annualised vol (×√252) range: {(daily_std.min()*np.sqrt(252)):.3f} … {(daily_std.max()*np.sqrt(252)):.3f}")


oc_returns_jp shape: (3796, 17)
  rows with ≥1 ticker |OC| > 50%: 0
  total cell-count of |OC| > 50%: 0
  no extreme OC return detected (>50%).

Daily OC return summary (Japan, 17 tickers):
  mean (daily) min / max     : -0.000909 / -0.000230
  std  (daily) min / max     : 0.006703 / 0.013759
  annualised vol (×√252) range: 0.106 … 0.218


## セル 7・8 の確認

| 系列 | 定義 | shape | NaN ポリシー | 用途 |
|---|---|---|---|---|
| `cc_returns` (全 26 銘柄) | `Close[t] / Close[t-1] - 1`、配当・分割調整後 | `(N_BIZ, 26)` | 先頭行 1 行のみ NaN（前日が無いため） | **PCA 推定**（共通因子の抽出）と米国側シグナル入力 |
| `oc_returns_jp` (日本 17 銘柄) | `Close[t] / Open[t] - 1`、同一営業日の調整後始値→終値 | `(N_BIZ, 17)` | NaN なし | **戦略 P/L 評価**（シグナル時点 t に対する翌日 t+1 の日中執行） |

- **配当・分割調整**：`auto_adjust=True` により Open・Close 双方が同一倍率で調整されるため、`Close / Open` の比では相殺され、追加処理不要（要件定義書 v3 §2.4）。
- **v2 から v3 への変更**：v2 では戦略 P/L を「日本側 CC リターン」で近似していたが、これは日本のオーバーナイトリターンを混入させ、戦略が狙う情報伝播構造を汚染するため廃止。v3 では OC のみを使う。
- **外れ値**：要件定義書 §3.3 に従い、±50% 超は警告ログのみ（停止しない）。出力で「`no extreme … detected`」を確認。

問題なければ SECTION 2C（事前相関 C_full とバックテストループ）に進む。
